In [ ]:
!nvidia-smi
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
!pip install -U transformers peft accelerate sentencepiece
!pip install llama-cpp-python -q

In [ ]:
!pip install bitsandbytes

In [1]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
LORA_PATH = "/kaggle/input/adapters-lora/adapters"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, LORA_PATH)

model = model.merge_and_unload()

model.save_pretrained("./merged_model")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.save_pretrained("./merged_model")

2026-01-16 09:05:54.486449: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768554354.509642     689 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768554354.516783     689 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768554354.535641     689 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768554354.535665     689 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768554354.535667     689 computation_placer.cc:177] computation placer alr

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

('./merged_model/tokenizer_config.json',
 './merged_model/special_tokens_map.json',
 './merged_model/chat_template.jinja',
 './merged_model/tokenizer.model',
 './merged_model/added_tokens.json',
 './merged_model/tokenizer.json')

In [2]:
!git clone https://github.com/ggerganov/llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 76139, done.
remote: Counting objects: 100% (224/224), done.
remote: Compressing objects: 100% (186/186), done.
remote: Total 76139 (delta 114), reused 39 (delta 38), pack-reused 75915 (from 4)
Receiving objects: 100% (76139/76139), 279.46 MiB | 37.62 MiB/s, done.
Resolving deltas: 100% (55240/55240), done.


In [3]:
%cd ./llama.cpp

/kaggle/working/llama.cpp


In [4]:
!cmake -B build
!cmake --build build --config Release -j

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

In [5]:
%cd ..

/kaggle/working


In [6]:
!python llama.cpp/convert_hf_to_gguf.py ./merged_model \
  --outfile model-q8_0.gguf \
  --outtype q8_0

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.bfloat16 --> Q8_0, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> Q8_0, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> Q8_0, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> Q8_0, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> Q8_0, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.bfloat16 --> Q8_0, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.

In [7]:
!python llama.cpp/convert_hf_to_gguf.py ./merged_model \
  --outfile model-bf16.gguf \
  --outtype bf16

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.bfloat16 --> BF16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> BF16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> BF16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> BF16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> BF16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.bfloat16 --> BF16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.

In [8]:
!python llama.cpp/convert_hf_to_gguf.py ./merged_model \
  --outfile model-f16.gguf \
  --outtype f16

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.bfloat16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.bfloat16 --> F16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_o

In [9]:
!ls

llama.cpp     model-bf16.gguf  model-q8_0.gguf
merged_model  model-f16.gguf   state.db


In [10]:
%cd llama.cpp

/kaggle/working/llama.cpp


In [11]:
%cd build

/kaggle/working/llama.cpp/build


In [12]:
%cd bin

/kaggle/working/llama.cpp/build/bin


In [13]:
ls

libggml-base.so@                llama-q8dot*
libggml-base.so.0@              llama-quantize*
libggml-base.so.0.9.5*          llama-qwen2vl-cli*
libggml-cpu.so@                 llama-retrieval*
libggml-cpu.so.0@               llama-save-load-state*
libggml-cpu.so.0.9.5*           llama-server*
libggml.so@                     llama-simple*
libggml.so.0@                   llama-simple-chat*
libggml.so.0.9.5*               llama-speculative*
libllama.so@                    llama-speculative-simple*
libllama.so.0@                  llama-tokenize*
libllama.so.0.0.7755*           llama-tts*
libmtmd.so@                     llama-vdot*
libmtmd.so.0@                   test-alloc*
libmtmd.so.0.0.7755*            test-arg-parser*
llama-batched*                  test-autorelease*
llama-batched-bench*            test-backend-ops*
llama-bench*                    test-backend-sampler*
llama-cli*                      test-barrier*
llama-completion*               test-c*
llama-convert-llama2c-to-ggml*  

In [14]:
%cd ..
%cd ..
%cd ..

/kaggle/working/llama.cpp/build
/kaggle/working/llama.cpp
/kaggle/working


In [15]:
!./llama.cpp/build/bin/llama-quantize model-bf16.gguf model-q4_0.gguf q4_0

main: build = 7755 (aa1dc3770)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing 'model-bf16.gguf' to 'model-q4_0.gguf' as Q4_0
llama_model_loader: direct I/O is enabled, disabling mmap
llama_model_loader: loaded meta data with 32 key-value pairs and 201 tensors from model-bf16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged_Model
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32          

In [16]:
import os

files = {
    "FP16 GGUF": "model-f16.gguf",
    "INT8 GGUF": "model-q8_0.gguf",
    "INT4 GGUF": "model-q4_0.gguf"
}

print(f"{'Format':<15} | {'Size (MB)':>10}")
print("-" * 30)

for name, path in files.items():
    size = os.path.getsize(path) / 1024**2
    print(f"{name:<15} | {size:>10.2f}")

Format          |  Size (MB)
------------------------------
FP16 GGUF       |    2099.05
INT8 GGUF       |    1115.62
INT4 GGUF       |     607.23


In [17]:
import os
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "./merged_model"
OUT_DIR = "./quantized"
os.makedirs(OUT_DIR, exist_ok=True)

def quantize(bits):
    print(f"\n--- INT{bits} Quantization ---")
    
    if bits == 8:
        bnb = BitsAndBytesConfig(load_in_8bit=True)
    else:
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16
        )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        quantization_config=bnb
    )

    save_path = f"{OUT_DIR}/model-int{bits}"
    model.save_pretrained(save_path)

    mem = model.get_memory_footprint() / 1024**2
    print(f"Memory footprint: {mem:.2f} MB")

    del model
    torch.cuda.empty_cache()

quantize(8)
quantize(4)


--- INT8 Quantization ---
Memory footprint: 1174.18 MB

--- INT4 Quantization ---
Memory footprint: 712.18 MB


In [18]:
from llama_cpp import Llama

llm = Llama(
    model_path="model-f16.gguf",
    n_ctx=2048,
    chat_format="llama-3",
    repetition_penalty=1.2,
    verbose = False

)

response = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "Answer the medical question accurately."},
        {"role": "user", "content": "What are the symptoms of Metastatic Squamous Neck Cancer with Occult Primary ?"}
    ],
    temperature=0.1,
    max_tokens=256
)

print(response["choices"][0]["message"]["content"])

Answer: The symptoms of metastatic squamous neck cancer with occult primary are similar to those of other types of cancer. They include:

- Pain or discomfort in the neck or face
- A lump or swelling in the neck or face
- A lump or swelling in the neck or face that does not respond to treatment
- A lump or swelling in the neck or face that is not responding to treatment
- A lump or swelling in the neck or face that is not responding to treatment and is not in the neck or face
- A lump or swelling in the neck or face that is not responding to treatment and is not in the neck or face
- A lump or swelling in the neck or face that is not responding to treatment and is not in the neck or face
- A lump or swelling in the neck or face that is not responding to treatment and is not in the neck or face
- A lump or swelling in the neck or face that is not responding to treatment and is not in the neck or face that is not in the neck or face
- A lump or swelling in the neck


In [19]:
import os

models = {
    "BF16": "./merged_model",
    "INT8 (bnb)": "./quantized/model-int8",
    "INT4 (bnb)": "./quantized/model-int4",
    "GGUF bf16": "model-bf16.gguf",
    "GGUF f16": "model-f16.gguf",
    "GGUF q8_0": "model-q8_0.gguf",
    "GGUF q4_0": "model-q4_0.gguf",
}

def get_size_mb(path):
    if os.path.isfile(path):
        return os.path.getsize(path) / 1024**2
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / 1024**2

print(f"{'Format':<15} | {'Size (MB)':>10}")
print("-" * 30)

for name, path in models.items():
    print(f"{name:<15} | {get_size_mb(path):>10.2f}")


Format          |  Size (MB)
------------------------------
BF16            |    2102.13
INT8 (bnb)      |    1175.73
INT4 (bnb)      |     727.13
GGUF bf16       |    2099.05
GGUF f16        |    2099.05
GGUF q8_0       |    1115.62
GGUF q4_0       |     607.23


In [20]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

prompt = "Explain the pathophysiology of endometriosis."

def benchmark_hf(model_path, label):
    tokenizer = AutoTokenizer.from_pretrained("./merged_model")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="cuda"
    )
    _ = model.generate(**inputs, max_new_tokens=1)
    torch.cuda.synchronize()

    start = time.time()

    output = model.generate(**inputs, max_new_tokens=128)

    # print(tokenizer.decode(output[0], skip_special_tokens = True))

    torch.cuda.synchronize()
    
    end = time.time()

    tokens = output.shape[-1]
    tps = tokens / (end - start)

    print(f"{label}: {tps:.2f} tokens/sec")

benchmark_hf("./merged_model", "BF16")
benchmark_hf("./quantized/model-int8", "INT8")
benchmark_hf("./quantized/model-int4", "INT4")


BF16: 40.48 tokens/sec
INT8: 10.75 tokens/sec
INT4: 18.30 tokens/sec


In [22]:
from llama_cpp import Llama
import time

def benchmark_gguf(path, label, prompt):
    llm = Llama(
        model_path=path,
        n_ctx=2048,
        n_threads=8,
        verbose=False
    )

    start = time.time()
    llm(prompt, max_tokens=128, echo= False)
    end = time.time()

    tps = 128 / (end - start)
    print(f"{label}: {tps:.2f} tokens/sec")
p1 = "Explain the pathophysiology of endometriosis."
p2 = "Daignosis to Chest pain."
benchmark_gguf("model-q8_0.gguf", "GGUF q8_0", p1)
benchmark_gguf("model-q4_0.gguf", "GGUF q4_0", p2)
benchmark_gguf("model-f16.gguf", "GGUF f16", p1)
benchmark_gguf("model-bf16.gguf", "GGUF bf16", p1)




GGUF q8_0: 11.18 tokens/sec
GGUF q4_0: 19.91 tokens/sec
GGUF f16: 7.83 tokens/sec
GGUF bf16: 7.74 tokens/sec
